In [59]:
# imports
from transformers import AutoProcessor, AutoModelForCausalLM
import torch
import os
from tqdm import tqdm
import shutil
import random
import pandas as pd
from glob import glob
import numpy as np
import seaborn as sns
from PIL import Image, ImageFilter, ImageEnhance
from pathlib import Path
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import matplotlib.image as mpimg
import cv2
from ultralytics import YOLO
import clip

# ML / evaluation (fusion classifier)
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, accuracy_score, classification_report
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

# Utils
from collections import defaultdict
from typing import List, Dict, Tuple, Optional


In [60]:
# Use GPU if available
print(torch.cuda.is_available())
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

True
Using device: cuda


In [61]:
clip_model, clip_preprocess = clip.load("ViT-B/32", device=device)
clip_model.eval()

CLIP(
  (visual): VisionTransformer(
    (conv1): Conv2d(3, 768, kernel_size=(32, 32), stride=(32, 32), bias=False)
    (ln_pre): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
    (transformer): Transformer(
      (resblocks): Sequential(
        (0): ResidualAttentionBlock(
          (attn): MultiheadAttention(
            (out_proj): NonDynamicallyQuantizableLinear(in_features=768, out_features=768, bias=True)
          )
          (ln_1): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
          (mlp): Sequential(
            (c_fc): Linear(in_features=768, out_features=3072, bias=True)
            (gelu): QuickGELU()
            (c_proj): Linear(in_features=3072, out_features=768, bias=True)
          )
          (ln_2): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
        )
        (1): ResidualAttentionBlock(
          (attn): MultiheadAttention(
            (out_proj): NonDynamicallyQuantizableLinear(in_features=768, out_features=768, bias=True)
          

In [62]:
yolo_model = YOLO("yolov8n.pt")

In [63]:
UCF_ROOT = Path(r"Datasets/UCF Crime Dataset")
SPLIT = "Train"  # "Train" or "Test"
NORMAL_CLASS = "NormalVideos"

split_dir = UCF_ROOT / SPLIT
print("UCF_ROOT exists:", UCF_ROOT.exists())
print("split_dir exists:", split_dir.exists())
print("classes:", [p.name for p in split_dir.iterdir() if p.is_dir()])

YOLO_DEVICE = 0 if torch.cuda.is_available() else "cpu" 

random.seed(42)
np.random.seed(42)
torch.manual_seed(42)

print("device:", device, "YOLO_DEVICE:", YOLO_DEVICE)


UCF_ROOT exists: True
split_dir exists: True
classes: ['Abuse', 'Arrest', 'Arson', 'Assault', 'Burglary', 'Explosion', 'Fighting', 'NormalVideos', 'RoadAccidents', 'Robbery', 'Shooting', 'Shoplifting', 'Stealing', 'Vandalism']
device: cuda YOLO_DEVICE: 0


In [64]:
NORMAL_PROMPTS = [
    "a normal street scene",
    "people walking normally",
    "a normal store scene",
    "a normal hallway scene",
]
ANOM_PROMPTS = [
    "a fight",
    "an assault",
    "a robbery",
    "a burglary",
    "an explosion",
    "a shooting",
    "arson",
    "vandalism",
    "a car accident",
    "shoplifting",
]

PROMPTS = NORMAL_PROMPTS + ANOM_PROMPTS

with torch.no_grad():
    text_tokens = clip.tokenize(PROMPTS).to(device)
    text_features = clip_model.encode_text(text_tokens)
    text_features = text_features / text_features.norm(dim=-1, keepdim=True)


In [65]:
def clip_prompt_scores(frame_bgr):
    img = cv2.cvtColor(frame_bgr, cv2.COLOR_BGR2RGB)
    img_pil = Image.fromarray(img)
    img_t = clip_preprocess(img_pil).unsqueeze(0).to(device)

    with torch.no_grad():
        img_feat = clip_model.encode_image(img_t)
        img_feat = img_feat / img_feat.norm(dim=-1, keepdim=True)
        sims = (img_feat @ text_features.T).squeeze(0)  # [num_prompts]
    return sims.float().cpu().numpy()

def yolo_basic_features(frame_bgr):
    rgb = cv2.cvtColor(frame_bgr, cv2.COLOR_BGR2RGB)
    r = yolo_model.predict(rgb, verbose=False, conf=0.25, device=YOLO_DEVICE)[0]

    if r.boxes is None or len(r.boxes) == 0:
        return np.array([0,0,0,0,0,0], dtype=np.float32)

    boxes = r.boxes
    conf = boxes.conf.detach().cpu().numpy()
    cls  = boxes.cls.detach().cpu().numpy().astype(int)
    xyxy = boxes.xyxy.detach().cpu().numpy()

    h, w = frame_bgr.shape[:2]
    areas = ((xyxy[:,2]-xyxy[:,0]) * (xyxy[:,3]-xyxy[:,1])) / (w*h + 1e-9)

    num_det   = len(conf)
    mean_conf = float(conf.mean())
    max_conf  = float(conf.max())
    mean_area = float(areas.mean())
    max_area  = float(areas.max())
    num_person = int((cls == 0).sum())  # COCO person=0

    return np.array([num_det, mean_conf, max_conf, mean_area, max_area, num_person], dtype=np.float32)


In [66]:
IMG_EXTS = {".png", ".jpg", ".jpeg", ".bmp"}

def list_images_in_dir(d: Path):
    return sorted([p for p in d.iterdir() if p.is_file() and p.suffix.lower() in IMG_EXTS])

def has_images(d: Path, max_checks=200) -> bool:
    try:
        for i, p in enumerate(d.iterdir()):
            if i >= max_checks:
                break
            if p.is_file() and p.suffix.lower() in IMG_EXTS:
                return True
    except Exception:
        return False
    return False

def list_clip_dirs_or_flat(class_dir: Path, max_scan=4000):
    """
    Returns either:
      - a list of clip folders (if subfolders exist with frames)
      - OR [class_dir] if it's a flat folder with frames directly inside
    """
    # flat case
    if has_images(class_dir):
        return [class_dir]

    # folder-of-folders case
    clips = []
    subdirs = [p for p in class_dir.iterdir() if p.is_dir()][:max_scan]
    for d in subdirs:
        if has_images(d):
            clips.append(d)
        else:
            sub2 = [p for p in d.iterdir() if p.is_dir()][:max_scan]
            for dd in sub2:
                if has_images(dd):
                    clips.append(dd)

    return sorted({c.resolve() for c in clips})

def read_frames_from_any_clip(clip_dir: Path, max_frames=60, start_idx=None):
    """
    Reads up to max_frames from clip_dir.
    If start_idx is provided, reads a consecutive chunk starting there.
    """
    imgs = list_images_in_dir(clip_dir)
    if len(imgs) == 0:
        return []

    if start_idx is None:
        # evenly sample across the whole folder
        stride = max(len(imgs) // max_frames, 1)
        chosen = imgs[::stride][:max_frames]
    else:
        start_idx = int(start_idx)
        chosen = imgs[start_idx : start_idx + max_frames]
        if len(chosen) < max_frames:
            # wrap if needed
            chosen = chosen + imgs[: max_frames - len(chosen)]

    frames = []
    for p in chosen:
        fr = cv2.imread(str(p))
        if fr is not None:
            frames.append(fr)
    return frames

def feature_from_clip_dir(clip_dir: Path, max_frames=60, start_idx=None):
    frames = read_frames_from_any_clip(clip_dir, max_frames=max_frames, start_idx=start_idx)
    if len(frames) == 0:
        return None

    feats = []
    for fr in frames:
        feats.append(np.concatenate([clip_prompt_scores(fr), yolo_basic_features(fr)], axis=0))

    feats = np.stack(feats, axis=0)
    return np.concatenate([feats.mean(axis=0), feats.max(axis=0)], axis=0)


In [67]:
N_NORMAL_SAMPLES = 10
N_ANOM_SAMPLES   = 10
MAX_FRAMES_PER_SAMPLE = 30 

split_dir = UCF_ROOT / SPLIT
class_dirs = [p for p in split_dir.iterdir() if p.is_dir()]

normal_dir = next((p for p in class_dirs if p.name.lower() == NORMAL_CLASS.lower()), None)
if normal_dir is None:
    raise RuntimeError(f"Couldn't find {NORMAL_CLASS}. Found: {[p.name for p in class_dirs]}")

anom_class_dirs = [p for p in class_dirs if p != normal_dir]

# Get clip dirs (or flat class dir)
normal_clip_dirs = list_clip_dirs_or_flat(normal_dir)
anom_clip_dirs = []
for d in anom_class_dirs:
    anom_clip_dirs += list_clip_dirs_or_flat(d)

random.shuffle(normal_clip_dirs)
random.shuffle(anom_clip_dirs)

print("Normal clip dirs (first 5):")
for p in normal_clip_dirs[:5]: print(" ", p)
print("\nAnomaly clip dirs (first 5):")
for p in anom_clip_dirs[:5]: print(" ", p)

def make_samples_from_clipdirs(clip_dirs, label, n_samples):
    samples = []
    # cycle through clip_dirs and create samples
    i = 0
    while len(samples) < n_samples and len(clip_dirs) > 0:
        cd = clip_dirs[i % len(clip_dirs)]
        imgs = list_images_in_dir(cd)
        if len(imgs) == 0:
            i += 1
            continue

        # choose a random start for a chunk
        start = random.randint(0, max(len(imgs) - 1, 0))
        samples.append((cd, start, label))
        i += 1
    return samples

normal_samples = make_samples_from_clipdirs(normal_clip_dirs, 0, N_NORMAL_SAMPLES)
anom_samples   = make_samples_from_clipdirs(anom_clip_dirs,   1, N_ANOM_SAMPLES)

print("\nNormal samples:", len(normal_samples), " Anom samples:", len(anom_samples))

X, y, names = [], [], []

for cd, start, lab in tqdm(normal_samples, desc="Normal samples"):
    vf = feature_from_clip_dir(cd, max_frames=MAX_FRAMES_PER_SAMPLE, start_idx=start)
    if vf is not None:
        X.append(vf); y.append(lab); names.append(f"{cd} | start={start}")

for cd, start, lab in tqdm(anom_samples, desc="Anomaly samples"):
    vf = feature_from_clip_dir(cd, max_frames=MAX_FRAMES_PER_SAMPLE, start_idx=start)
    if vf is not None:
        X.append(vf); y.append(lab); names.append(f"{cd} | start={start}")

if len(X) < 4:
    raise RuntimeError("Still not enough samples extracted. Something is wrong with reading png frames.")

X = np.stack(X, axis=0)
y = np.array(y, dtype=np.int64)

print("X shape:", X.shape, "y counts:", {0:int((y==0).sum()), 1:int((y==1).sum())})

Normal clip dirs (first 5):
  Datasets\UCF Crime Dataset\Train\NormalVideos

Anomaly clip dirs (first 5):
  Datasets\UCF Crime Dataset\Train\RoadAccidents
  Datasets\UCF Crime Dataset\Train\Fighting
  Datasets\UCF Crime Dataset\Train\Arson
  Datasets\UCF Crime Dataset\Train\Shooting
  Datasets\UCF Crime Dataset\Train\Vandalism

Normal samples: 10  Anom samples: 10


Anomaly samples: 100%|██████████| 10/10 [00:10<00:00,  1.01s/it]

X shape: (20, 40) y counts: {0: 10, 1: 10}


In [ ]:
test_size = 0.33 if len(y) >= 6 else 0.5

try:
    X_tr, X_te, y_tr, y_te = train_test_split(
        X, y, test_size=test_size, random_state=42, stratify=y
    )
except Exception:
    X_tr, X_te, y_tr, y_te = train_test_split(
        X, y, test_size=test_size, random_state=42
    )

clf = Pipeline([
    ("scaler", StandardScaler()),
    ("lr", LogisticRegression(max_iter=2000))
])

clf.fit(X_tr, y_tr)

probs = clf.predict_proba(X_te)[:, 1]
preds = (probs >= 0.5).astype(int)

print("Acc:", accuracy_score(y_te, preds))
if len(np.unique(y_te)) == 2:
    print("AUC:", roc_auc_score(y_te, probs))
print(classification_report(y_te, preds))


Acc: 0.42857142857142855
AUC: 0.75
              precision    recall  f1-score   support

           0       0.50      0.25      0.33         4
           1       0.40      0.67      0.50         3

    accuracy                           0.43         7
   macro avg       0.45      0.46      0.42         7
weighted avg       0.46      0.43      0.40         7



In [69]:
# show which prompts were strongest on ONE random anomaly SAMPLE (first frame only)

if len(anom_samples) == 0:
    print("No anomaly samples available.")
else:
    cd, start, lab = anom_samples[0]   # (clip_dir, start_idx, label)

    frames = read_frames_from_any_clip(cd, max_frames=1, start_idx=start)
    if frames:
        sims = clip_prompt_scores(frames[0])
        top_idx = np.argsort(-sims)[:8]
        print("Top prompts for:", cd, "| start=", start)
        for i in top_idx:
            print(f"{PROMPTS[i]:<30}  score={sims[i]:.4f}")
    else:
        print("Could not read frames from:", cd)


Top prompts for: Datasets\UCF Crime Dataset\Train\RoadAccidents | start= 23462
a car accident                  score=0.2510
people walking normally         score=0.2484
a normal street scene           score=0.2379
a shooting                      score=0.2329
a robbery                       score=0.2172
a fight                         score=0.2152
an assault                      score=0.2129
a burglary                      score=0.2053
